# Multi-Seed Variance Check

Input: `data/processed/windows_cc1/X_*.npy`. Retrains the VAE with the exact
validated architecture/hyperparameters from `train_vae.ipynb`
(`latent_dim=32, hidden1=64, hidden2=32, beta_max=0.01, warmup=10 epochs`) across
**3 random seeds** (42 — the original reported seed — plus 7 and 123), evaluates
each fully-trained model the same way `vae_eval.ipynb` does, and reports
mean ± std for AUC-ROC/AUC-PR/F1 per set.

**Why this notebook exists:** every number reported so far (train_vae.ipynb,
vae_eval.ipynb, adaptive_threshold*.ipynb) came from ONE trained model
(seed=42). Training is known to be non-deterministic — the ablation numbers
visibly shifted between reruns earlier in this project. Without a variance
estimate, there's no way to know whether the reported F1/AUC differences
between sets (or methods) reflect genuine signal or training noise.

**Scope note:** 3 seeds, not 10+, given the compute cost of full training
(~10-15 min each on CPU) — a legitimate minimum for a resource-constrained
project, not a substitute for more seeds if time allows later. State this
limitation explicitly if reporting results in the thesis.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle, os, time
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')

DEVICE = torch.device('cpu')
SEEDS = [42, 7, 123]

HIDDEN1, HIDDEN2, LATENT_DIM = 64, 32, 32
BETA_MAX, WARMUP_EPOCHS = 0.01, 10
FULL_MAX_EPOCHS, FULL_PATIENCE = 300, 20
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0

DRIFT_SETS = ['drift_sc1', 'drift_sc2', 'drift_cc2']
ALL_SETS   = ['cc1_test'] + DRIFT_SETS

raw = {name: np.load(os.path.join(DATA_DIR, f'X_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
labels = {name: np.load(os.path.join(DATA_DIR, f'y_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
data = {name: np.clip(X, -CLIP, CLIP).astype(np.float32) for name, X in raw.items()}
INPUT_DIM = data['cc1_train'].shape[1]

X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
print(f'INPUT_DIM={INPUT_DIM}  SEEDS={SEEDS}')

INPUT_DIM=26  SEEDS=[42, 7, 123]


## VAE class + training loop (identical to `train_vae.ipynb`, post-KL-fix)

In [2]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon_loss + beta * kl, recon_loss, kl

def train_one_seed(seed, max_epochs=FULL_MAX_EPOCHS, patience=FULL_PATIENCE):
    torch.manual_seed(seed); np.random.seed(seed)
    model = VAE(INPUT_DIM, HIDDEN1, HIDDEN2, LATENT_DIM).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
    best_val, best_state, patience_ctr = float('inf'), None, 0

    for epoch in range(max_epochs):
        beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * BETA_MAX
        model.train()
        for (xb,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss, _, _ = vae_loss(recon, xb, mu, logvar, beta)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(X_val_t)
            vloss, _, _ = vae_loss(recon, X_val_t, mu, logvar, beta)
        if vloss.item() < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, epoch + 1

print('Training function defined.')

Training function defined.


## Train + evaluate each seed

In [3]:
per_seed_results = {}
for seed in SEEDS:
    print(f'--- seed={seed} ---')
    t0 = time.time()
    model, best_val, n_epochs = train_one_seed(seed)
    elapsed = time.time() - t0
    print(f'  trained: {n_epochs} epochs, best_val_loss={best_val:.4f}  ({elapsed:.0f}s)')

    with torch.no_grad():
        mse_train = model.anomaly_score(X_train_t).numpy()
        mse_val   = model.anomaly_score(X_val_t).numpy()
    val_p99 = float(np.percentile(mse_val, 99))

    seed_result = {'best_val_loss': best_val, 'n_epochs': n_epochs, 'val_p99': val_p99}
    for name in ALL_SETS:
        X_t = torch.from_numpy(data[name])
        with torch.no_grad():
            mse = model.anomaly_score(X_t).numpy()
        y_true = labels[name]
        auc_roc = roc_auc_score(y_true, mse)
        auc_pr  = average_precision_score(y_true, mse)
        pred = (mse > val_p99).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred, zero_division=0)
        seed_result[name] = {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'f1': f1, 'precision': prec, 'recall': rec}
        print(f'    {name:10s}  AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  F1={f1:.4f}')

    per_seed_results[seed] = seed_result
    print()

--- seed=42 ---
  trained: 300 epochs, best_val_loss=0.2320  (1161s)
    cc1_test    AUC-ROC=0.8763  AUC-PR=0.6014  F1=0.6175
    drift_sc1   AUC-ROC=0.8515  AUC-PR=0.0181  F1=0.0399
    drift_sc2   AUC-ROC=0.6327  AUC-PR=0.0615  F1=0.0235
    drift_cc2   AUC-ROC=0.8812  AUC-PR=0.4089  F1=0.2757

--- seed=7 ---
  trained: 255 epochs, best_val_loss=0.2503  (1013s)
    cc1_test    AUC-ROC=0.8862  AUC-PR=0.5617  F1=0.4573
    drift_sc1   AUC-ROC=0.8880  AUC-PR=0.0445  F1=0.0458
    drift_sc2   AUC-ROC=0.5884  AUC-PR=0.0669  F1=0.0170
    drift_cc2   AUC-ROC=0.8941  AUC-PR=0.4228  F1=0.1619

--- seed=123 ---
  trained: 300 epochs, best_val_loss=0.2421  (1864s)
    cc1_test    AUC-ROC=0.8591  AUC-PR=0.5946  F1=0.5594
    drift_sc1   AUC-ROC=0.8899  AUC-PR=0.1293  F1=0.0467
    drift_sc2   AUC-ROC=0.5882  AUC-PR=0.1558  F1=0.0214
    drift_cc2   AUC-ROC=0.8946  AUC-PR=0.4299  F1=0.1765



## Aggregate: mean ± std across seeds, per set

In [4]:
print(f'{"set":12s} {"AUC-ROC":>16s} {"AUC-PR":>16s} {"F1":>16s}')
summary = {}
for name in ALL_SETS:
    aucs   = [per_seed_results[s][name]['auc_roc'] for s in SEEDS]
    aucprs = [per_seed_results[s][name]['auc_pr'] for s in SEEDS]
    f1s    = [per_seed_results[s][name]['f1'] for s in SEEDS]
    summary[name] = {
        'auc_roc_mean': np.mean(aucs), 'auc_roc_std': np.std(aucs),
        'auc_pr_mean':  np.mean(aucprs), 'auc_pr_std':  np.std(aucprs),
        'f1_mean':      np.mean(f1s),   'f1_std':      np.std(f1s),
        'f1_values':    f1s,
    }
    print(f'{name:12s} {np.mean(aucs):.4f}+/-{np.std(aucs):.4f}   {np.mean(aucprs):.4f}+/-{np.std(aucprs):.4f}   {np.mean(f1s):.4f}+/-{np.std(f1s):.4f}')

print()
print('Individual per-seed F1 values (for reference):')
for name in ALL_SETS:
    print(f'  {name:12s}: {[round(v, 3) for v in summary[name]["f1_values"]]}')

set                   AUC-ROC           AUC-PR               F1
cc1_test     0.8739+/-0.0112   0.5859+/-0.0173   0.5448+/-0.0662
drift_sc1    0.8765+/-0.0177   0.0640+/-0.0474   0.0441+/-0.0030
drift_sc2    0.6031+/-0.0209   0.0947+/-0.0432   0.0206+/-0.0027
drift_cc2    0.8900+/-0.0062   0.4206+/-0.0087   0.2047+/-0.0505

Individual per-seed F1 values (for reference):
  cc1_test    : [0.618, 0.457, 0.559]
  drift_sc1   : [0.04, 0.046, 0.047]
  drift_sc2   : [0.024, 0.017, 0.021]
  drift_cc2   : [0.276, 0.162, 0.176]


## Save results

In [5]:
out_path = os.path.join(MODEL_DIR, 'multi_seed_variance.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'seeds': SEEDS, 'per_seed': per_seed_results, 'summary': summary}, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\multi_seed_variance.pkl


## How to read this

The original reported numbers (seed=42) should fall within, or close to, the
mean±std range shown here. A small std relative to the mean (e.g. std <10% of
mean) means the headline results are stable and not a lucky/unlucky training
run. A large std is itself an important, honest finding — it would mean any
single-seed number (including the ones already reported) should be quoted with
this uncertainty, not as a precise point estimate.